# 📊 DocuMind AI — RAG Pipeline Analysis
**Business Understanding:** Analyze the RAG pipeline performance — embedding quality, retrieval accuracy, chunk distribution

This notebook covers:
1. PDF Processing Analysis (chunk size distribution, page coverage)
2. Embedding Quality (vector space visualization with PCA/TSNE)
3. Retrieval Performance (similarity score distributions)
4. LLM Response Quality (grounding rate, confidence distribution)
5. System Benchmarking (latency per stage)

In [1]:
# ── Setup ──────────────────────────────────────────────
import sys
sys.path.append('..')   # add project root to path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})

print('✅ Libraries loaded!')

✅ Libraries loaded!


## 1️⃣ PDF Processing — Chunk Analysis

In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
from backend.services.pdf_processor import PDFProcessor
from backend.core import settings
from pathlib import Path
# ── Process a sample PDF ──
# Place any PDF in data/uploads/ first
upload_dir = Path('../data/uploads')   # ✅ folder, not file
pdf_files  = list(upload_dir.glob('*.pdf'))

if not pdf_files:
    print('⚠️  No PDFs found in data/uploads/')
    print('   Upload a PDF via the web UI first, then re-run this notebook.')
else:
    processor = PDFProcessor()
    doc = processor.process(str(pdf_files[0]))

    print(f'📄 Document: {doc.filename}')
    print(f'   Pages     : {doc.total_pages}')
    print(f'   Chunks    : {doc.total_chunks}')
    print(f'   File Size : {doc.file_size_kb:.1f} KB')
    print(f'   Avg chunk : {np.mean([c.char_count for c in doc.chunks]):.0f} chars')

⚠️  No PDFs found in data/uploads/
   Upload a PDF via the web UI first, then re-run this notebook.


In [ ]:
if pdf_files:
    chunk_sizes = [c.char_count for c in doc.chunks]
    pages       = [c.page_number for c in doc.chunks]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Chunk size distribution
    axes[0].hist(chunk_sizes, bins=30, color='#2563eb', edgecolor='white', linewidth=0.5)
    axes[0].axvline(settings.chunk_size, color='red', linestyle='--', label=f'Max chunk size ({settings.chunk_size})')
    axes[0].set_xlabel('Characters per chunk')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Chunk Size Distribution', fontweight='bold')
    axes[0].legend(fontsize=10)

    # Chunks per page
    page_counts = pd.Series(pages).value_counts().sort_index()
    axes[1].bar(page_counts.index, page_counts.values, color='#16a34a', edgecolor='white', linewidth=0.5)
    axes[1].set_xlabel('Page Number')
    axes[1].set_ylabel('Chunks')
    axes[1].set_title('Chunks per Page', fontweight='bold')

    # Cumulative coverage
    cumulative = np.cumsum(sorted(chunk_sizes, reverse=True))
    axes[2].plot(range(1, len(cumulative)+1), cumulative / sum(chunk_sizes) * 100,
                 color='#7c3aed', linewidth=2)
    axes[2].set_xlabel('Number of chunks')
    axes[2].set_ylabel('% of total content')
    axes[2].set_title('Content Coverage Curve', fontweight='bold')
    axes[2].grid(True, alpha=0.4)

    plt.suptitle(f'PDF Processing Analysis — {doc.filename}', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../docs/chunk_analysis.png', dpi=120, bbox_inches='tight')
    plt.show()

## 2️⃣ Embedding Quality — Vector Space Visualization

In [ ]:
from backend.services.embeddings import get_embedding_service

if pdf_files:
    embedding_svc = get_embedding_service()
    print(f'Embedding model : {settings.embedding_model}')
    print(f'Vector dimension: {embedding_svc.vector_dimension}')

    # Embed all chunks
    texts = [c.text for c in doc.chunks[:100]]  # sample 100 for speed
    print(f'\nEmbedding {len(texts)} chunks...')
    embeddings = embedding_svc.embed_texts(texts)
    print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
if pdf_files and len(texts) >= 10:
    # PCA to 2D for visualization
    pca = PCA(n_components=2, random_state=42)
    coords_pca = pca.fit_transform(embeddings)

    # Color by page number
    page_nums = [doc.chunks[i].page_number for i in range(len(texts))]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # PCA plot
    sc = axes[0].scatter(coords_pca[:,0], coords_pca[:,1],
                         c=page_nums, cmap='viridis', alpha=0.7, s=40)
    plt.colorbar(sc, ax=axes[0], label='Page Number')
    axes[0].set_title('PCA — Chunk Embeddings (colored by page)', fontweight='bold')
    axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
    axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')

    # Cosine similarity heatmap (first 20 chunks)
    sim_matrix = embeddings[:20] @ embeddings[:20].T
    sns.heatmap(sim_matrix, ax=axes[1], cmap='Blues', vmin=0, vmax=1,
                xticklabels=False, yticklabels=False, cbar_kws={'label': 'Cosine Similarity'})
    axes[1].set_title('Chunk-to-Chunk Similarity Matrix (first 20)', fontweight='bold')
    axes[1].set_xlabel('Chunk index')
    axes[1].set_ylabel('Chunk index')

    plt.suptitle('Embedding Space Analysis', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('../docs/embedding_analysis.png', dpi=120, bbox_inches='tight')
    plt.show()

    explained = pca.explained_variance_ratio_.sum() * 100
    print(f'\nPCA Info: 2 components explain {explained:.1f}% of variance')

## 3️⃣ Retrieval Performance — Similarity Score Analysis

In [ ]:
from backend.services.vector_store import get_vector_store

# Sample queries to test retrieval quality
sample_queries = [
    'What is the main objective?',
    'What are the key findings?',
    'What methodology was used?',
    'What are the conclusions?',
    'What are the recommendations?',
]

store = get_vector_store()
all_docs = store.list_documents()

if not all_docs:
    print('⚠️  No documents indexed. Run the upload step first.')
else:
    results_data = []
    for q in sample_queries:
        results = store.search(q, top_k=5)
        for r in results:
            results_data.append({
                'query':      q,
                'similarity': r['similarity'],
                'page':       r['page_number'],
            })

    df_results = pd.DataFrame(results_data)

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Similarity distribution per query
    for q in sample_queries:
        sims = df_results[df_results['query'] == q]['similarity']
        axes[0].plot(range(len(sims)), sorted(sims, reverse=True),
                     marker='o', linewidth=2, markersize=5,
                     label=q[:35]+'...' if len(q)>35 else q)
    axes[0].set_xlabel('Rank (1 = most similar)')
    axes[0].set_ylabel('Cosine Similarity Score')
    axes[0].set_title('Retrieval Similarity by Rank', fontweight='bold')
    axes[0].legend(fontsize=9, loc='upper right')
    axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='0.5 threshold')

    # Box plot of similarity scores
    df_results.boxplot(column='similarity', by='query', ax=axes[1],
                       grid=True, vert=False)
    axes[1].set_title('Similarity Score Distribution by Query', fontweight='bold')
    axes[1].set_xlabel('Cosine Similarity')
    axes[1].set_ylabel('')
    plt.sca(axes[1])
    plt.yticks(fontsize=8)

    plt.suptitle('Retrieval Performance Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../docs/retrieval_analysis.png', dpi=120, bbox_inches='tight')
    plt.show()

    print('\nMean similarity scores per query:')
    print(df_results.groupby('query')['similarity'].mean().round(3).to_string())

## 4️⃣ End-to-End RAG Pipeline Benchmark

In [ ]:
import time
from backend.services.rag_engine import get_rag_engine

engine = get_rag_engine()

benchmark_queries = [
    'What is this document about?',
    'List the main points covered.',
    'What data or evidence is presented?',
]

benchmark_results = []

if all_docs:
    print('Running RAG pipeline benchmark...')
    for q in benchmark_queries:
        response = engine.query(q)
        benchmark_results.append({
            'query':           q,
            'latency_ms':      response.latency_ms,
            'chunks':          response.chunks_retrieved,
            'confidence':      response.confidence,
            'is_grounded':     response.is_grounded,
            'answer_length':   len(response.answer),
        })
        print(f'  ✅ {q[:50]} | {response.latency_ms:.0f}ms | {response.confidence}')

    df_bench = pd.DataFrame(benchmark_results)
    print('\n=== Benchmark Summary ===')
    print(df_bench[['query','latency_ms','confidence','is_grounded']].to_string(index=False))
    print(f'\nAvg latency : {df_bench["latency_ms"].mean():.0f}ms')
    print(f'Grounded rate: {df_bench["is_grounded"].mean()*100:.0f}%')

In [ ]:
if benchmark_results:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # Latency bar chart
    colors = ['#2563eb', '#16a34a', '#d97706']
    axes[0].barh(range(len(df_bench)), df_bench['latency_ms'], color=colors)
    axes[0].set_yticks(range(len(df_bench)))
    axes[0].set_yticklabels([q[:25]+'...' for q in df_bench['query']], fontsize=9)
    axes[0].set_xlabel('Latency (ms)')
    axes[0].set_title('End-to-End Latency per Query', fontweight='bold')

    # Confidence distribution
    conf_counts = df_bench['confidence'].value_counts()
    conf_colors = {'HIGH':'#16a34a','MEDIUM':'#d97706','LOW':'#dc2626'}
    bar_colors  = [conf_colors.get(c,'#888') for c in conf_counts.index]
    axes[1].bar(conf_counts.index, conf_counts.values, color=bar_colors, edgecolor='white')
    axes[1].set_title('Confidence Distribution', fontweight='bold')
    axes[1].set_ylabel('Count')

    # Answer length vs latency
    axes[2].scatter(df_bench['answer_length'], df_bench['latency_ms'],
                    s=100, color='#7c3aed', alpha=0.8)
    for i, row in df_bench.iterrows():
        axes[2].annotate(f'Q{i+1}',
                         (row['answer_length'], row['latency_ms']),
                         textcoords='offset points', xytext=(5,5), fontsize=9)
    axes[2].set_xlabel('Answer length (chars)')
    axes[2].set_ylabel('Latency (ms)')
    axes[2].set_title('Answer Length vs Latency', fontweight='bold')

    plt.suptitle('RAG Pipeline Benchmark', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../docs/benchmark.png', dpi=120, bbox_inches='tight')
    plt.show()